# Model trajanja — backtest (Etapa 11a)

Hijerarhijski log-normalni model (`src/lib/estimates/model.ts`, isti u `hoot_ml/duration.py`) prema starom blendu `k/(k+5)`. Vremenski backtest: svako mjerenje se predviđa iz mjerenja prije njega. Metrike: MAE na log-minutama (tačka = medijan) i pokrivenost P80 za jednu jedinicu (cilj ~0,8).

Bez izvoza (`ml/data/`) notebook radi na sintetičkim podacima iz samog modela.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from hoot_ml.data import load
from hoot_ml.duration import load_priors
from hoot_ml.duration_fit import (
    MIN_FIT_SESSIONS,
    backtest,
    fit,
    samples_from_export,
    synthetic_samples,
)

data = Path("../data")
real = data.exists() and any(data.glob("*.parquet"))
samples = samples_from_export(load(data)) if real else synthetic_samples(0)
print("stvarni podaci" if real else "sintetički podaci", len(samples), "mjerenja")

## Backtest

In [ ]:
priors = load_priors()
result = backtest(samples, priors)
result

In [ ]:
# Synthetic: many seeds, for the spread of the difference.
runs = [backtest(synthetic_samples(seed), priors) for seed in range(50)]
diff = np.array([r.mae_model - r.mae_old for r in runs])
pd.Series(
    {
        "mae_model": np.mean([r.mae_model for r in runs]),
        "mae_old": np.mean([r.mae_old for r in runs]),
        "diff": diff.mean(),
        "diff_se": diff.std() / np.sqrt(len(diff)),
        "coverage_p80": np.mean([r.coverage_p80 for r in runs]),
    }
).round(4)

## Fit priora

Novi `priors.json` (nova `version` → `input_hash`) tek od dovoljno sesija; s jednim korisnikom se fituju σ i τ_izvor, τ_korisnik ostaje.

In [ ]:
sessions = samples["session_id"].nunique()
if sessions >= MIN_FIT_SESSIONS:
    fitted = fit(samples, priors)
    print("sigma", round(fitted.sigma, 3), "tauSource", round(fitted.tauSource, 3))
    print(backtest(samples, fitted))
else:
    print(f"{sessions} sesija < {MIN_FIT_SESSIONS}: prior ostaje")